In [ ]:
import pandas as pd

input_path = "/content/sentiment-analysis.csv"
output_path = "/content/cleaned_sentimentData.csv"
texts = []
sentiments = []


with open(input_path, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue

        parts = line.split(",", 2)
        if len(parts) >= 2:
            text = parts[0].strip().strip('"')
            sentiment = parts[1].strip()


            if text.lower() == "text" and sentiment.lower() == "sentiment":
                continue

            texts.append(text)
            sentiments.append(sentiment)

df = pd.DataFrame({
    "Text": texts,
    "Sentiment": sentiments
})

print(df.head())
print("-> Number of rows:", len(df)+1)
df.to_csv(output_path, index=False)
print(f"Cleaned dataset is saved to: {output_path}")


In [ ]:
import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset, random_split
from transformers import BertTokenizer, BertForSequenceClassification
from torch.optim import AdamW
from tqdm import tqdm

# Load dataset
df = pd.read_csv("/content/cleaned_sentimentData.csv", sep=",")
df.columns = df.columns.str.strip()

# Label mapping
label_map = {"negative": 0, "neutral": 1, "positive": 2}
df["label"] = df["Sentiment"].str.lower().map(label_map)

# Tokenization
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
encodings = tokenizer(
    df["Text"].tolist(),
    truncation=True,
    padding=True,
    max_length=128,
    return_tensors="pt"
)

# Dataset preparation
input_ids = encodings["input_ids"]
attention_mask = encodings["attention_mask"]
labels = torch.tensor(df["label"].values)
dataset = TensorDataset(input_ids, attention_mask, labels)

# Split dataset
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)

# Model and optimizer
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=3)
optimizer = AdamW(model.parameters(), lr=2e-5)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Training
epochs = 40
best_val_loss = float('inf')

for epoch in range(epochs):
    model.train()
    total_loss = 0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
        b_input_ids, b_attention_mask, b_labels = [x.to(device) for x in batch]
        model.zero_grad()
        outputs = model(b_input_ids, attention_mask=b_attention_mask, labels=b_labels)
        loss = outputs.loss
        total_loss += loss.item()
        loss.backward()
        optimizer.step()

    avg_train_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1} - Training loss: {avg_train_loss:.4f}")


    model.eval()
    val_loss = 0
    with torch.no_grad():
        for batch in val_loader:
            b_input_ids, b_attention_mask, b_labels = [x.to(device) for x in batch]
            outputs = model(b_input_ids, attention_mask=b_attention_mask, labels=b_labels)
            val_loss += outputs.loss.item()

    avg_val_loss = val_loss / len(val_loader)
    print(f"Epoch {epoch+1} - Validation loss: {avg_val_loss:.4f}")

    # ---- Save best model ----
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), "sentiment_bestmodel.pt")
        print(f"-> Best model saved at epoch {epoch+1} with val_loss = {avg_val_loss:.4f}")

print("Fine-tuning complete!")


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Epoch 1/40: 100%|██████████| 5/5 [00:00<00:00,  9.98it/s]


Epoch 1 - Training loss: 1.0723
Epoch 1 - Validation loss: 0.7921
-> Best model saved at epoch 1 with val_loss = 0.7921


Epoch 2/40: 100%|██████████| 5/5 [00:00<00:00, 12.49it/s]


Epoch 2 - Training loss: 0.7717
Epoch 2 - Validation loss: 0.5863
-> Best model saved at epoch 2 with val_loss = 0.5863


Epoch 3/40: 100%|██████████| 5/5 [00:00<00:00, 10.06it/s]


...

Epoch 38 - Training loss: 0.0048
Epoch 38 - Validation loss: 0.0040
-> Best model saved at epoch 38 with val_loss = 0.0040


Epoch 39/40: 100%|██████████| 5/5 [00:00<00:00,  9.54it/s]


Epoch 39 - Training loss: 0.0046
Epoch 39 - Validation loss: 0.0038
-> Best model saved at epoch 39 with val_loss = 0.0038


Epoch 40/40: 100%|██████████| 5/5 [00:00<00:00, 10.34it/s]


Epoch 40 - Training loss: 0.0045
Epoch 40 - Validation loss: 0.0036
-> Best model saved at epoch 40 with val_loss = 0.0036
Fine-tuning complete!


In [ ]:
import torch
from transformers import BertTokenizer, BertForSequenceClassification

model_name = "bert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(model_name)

model = BertForSequenceClassification.from_pretrained(model_name, num_labels=3)

model.load_state_dict(torch.load("/content/sentiment_bestmodel.pt", map_location=torch.device("cpu")))
model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

id2label = {0: "Negative", 1: "Neutral", 2: "Positive"}

def predict_sentiment(texts):
    """
    Predicts sentiment for a list of input texts.
    """
    # Tokenizing input
    encodings = tokenizer(
        texts,
        truncation=True,
        padding=True,
        max_length=128,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = model(**encodings)
        logits = outputs.logits
        predictions = torch.argmax(logits, dim=1).cpu().numpy()

    return [id2label[p] for p in predictions]

# ---- Test the model ----
sample_texts = [
    "I absolutely loved the service, it was excellent!",
    "The food was okay, nothing special.",
    "I'm really disappointed with the product quality."
]

predictions = predict_sentiment(sample_texts)


for text, pred in zip(sample_texts, predictions):
    print(f"Text: {text}\nPredicted Sentiment: {pred}\n")


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Text: I absolutely loved the service, it was excellent!
Predicted Sentiment: Positive

Text: The food was okay, nothing special.
Predicted Sentiment: Negative

Text: I'm really disappointed with the product quality.
Predicted Sentiment: Negative

